# math_agent debug

After the `magic_number` fixes, does `math_agent` (basic_agent + calculator on GSM8K) learn?

Applied the same reward-structure changes: `valid_submit` (requires numeric arg), `tool_call_failures` penalty, dropped submit weight from 1.5 → 0.2.

No `react`-specific fixes needed — `basic_agent` preserves `tool_calls` in final state.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_HOME"] = "/opt/nvidia/hpc_sdk/Linux_aarch64/24.11/cuda/12.6"

import torch, trl, inspect_ai, transformers
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
print(f"trl: {trl.__version__} | inspect-ai: {inspect_ai.__version__} | transformers: {transformers.__version__}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {free//1024**3}GB free / {total//1024**3}GB")

In [ ]:
# vLLM health + eval sanity: run math_agent task on 4 base-model samples and
# confirm the new scorers fire as expected.
import httpx
from inspect_ai import eval as inspect_eval, task_with
from inspect_ai.dataset import MemoryDataset
from inspect_ai.model import GenerateConfig, get_model
from transformers import AutoTokenizer
import inspect_rl.trl_vllm_provider  # noqa: F401
from inspect_rl.example.math_agent import get_task

print(f"vLLM health: {httpx.get('http://localhost:8000/health/', timeout=5).json()}")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
task = get_task(split="train")
samples = list(task.dataset)[:4]
print(f"scorers: {[s.__qualname__.split('.')[0] for s in task.scorer]}")
print(f"sample[0]: q={samples[0].input[0].text[:80]!r}  target={samples[0].target!r}")

model = get_model(
    f"trl-vllm/{MODEL}", base_url="http://localhost:8000", tokenizer=tok,
    config=GenerateConfig(max_tokens=1024, temperature=1.0),
    batch_timeout=1.0, max_batch_size=4,
)
logs = inspect_eval(
    task_with(task, dataset=MemoryDataset(samples=samples), epochs=1),
    model=model, max_samples=4, display="plain", fail_on_error=False,
)
log = logs[0]
for s in (log.samples or []):
    scores = {n: sc.value for n, sc in (s.scores or {}).items()}
    print(f"  sample {s.id}: output={s.output.completion[:30]!r:30}  scores={scores}")

In [ ]:
import httpx, os

# Re-enable wandb for this run (cell 1 turned it off).
os.environ.pop("WANDB_DISABLED", None)
os.environ.setdefault("WANDB_PROJECT", "inspect-rl")
os.environ.setdefault("WANDB_ENTITY", os.environ.get("USER", "unknown").replace(".", "-").lower())
print(f"wandb → {os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")

try: httpx.post("http://localhost:8000/close_communicator/", json={}, timeout=5)
except Exception as e: print(f"close_communicator: {e}")
print("NCCL reset ok")

In [ ]:
# 50-step training, wandb on. max_completion_length=512 (base eval showed
# ~90 tokens/sample — 512 is plenty and halves the activation memory vs 1024
# which OOM'd here). PYTORCH_ALLOC_CONF mitigates fragmentation between
# rollout generate calls and training fwd passes.
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import GRPOConfig
from inspect_rl.trainer import inspect_rl_train

task_train = get_task(split="train")
task_eval = get_task(split="test")

grpo_config = GRPOConfig(
    output_dir="outputs/math_agent_debug",
    max_steps=50,
    per_device_train_batch_size=8,
    num_generations=8,
    max_completion_length=512,
    temperature=1.0,
    learning_rate=5e-6,
    warmup_steps=10,
    bf16=True,
    fp16=False,
    report_to="wandb",
    run_name="math_agent_debug_50steps",
    save_steps=100,
    logging_steps=1,
    use_vllm=True,
    vllm_mode="server",
    vllm_server_host="localhost",
    vllm_server_port=8000,
    # KL to base — bs=8,num_gen=8 gives 1 prompt/grad step, so signal is
    # narrow and gradient noise drifts the policy. beta=0.01 keeps style
    # close to base without choking the reward gradient.
    beta=0.01,
    # matches Task.scorer: [correctness, valid_submit, tool_call_failures]
    reward_weights=[4.0, 0.2, -0.3],
)
print(f"Training {MODEL} for {grpo_config.max_steps} steps…")
trainer = inspect_rl_train(
    task=task_train,
    model=MODEL,
    grpo_config=grpo_config,
    vllm_base_url="http://localhost:8000",
    dataset_limit=500,
    eval_task=task_eval,
    eval_steps=10,
    eval_limit=32,
)
print("Training complete.")
